In [1]:
from dotenv import load_dotenv
import os

In [2]:
import requests

In [3]:
load_dotenv()
socks_proxy = os.getenv('PROXY')


In [4]:
session = requests.Session()
user_agent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_3) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/72.0.3626.121 Safari/537.36'
headers = {'user-agent' : user_agent}
proxies = {
    'http' : f'socks5h://{socks_proxy}',
    'https' : f'socks5h://{socks_proxy}',
}

locationIdResponse = session.get('https://sigmas.social.gouv.fr/server/rest/services/baignades/fra_vue_baignade/MapServer/5/query?f=json&where=1%3D1&returnIdsOnly=true&geometry=-9678710.360800,-28545813.431743,30099965.362980,42759085.476384&geometryType=esriGeometryEnvelope&spatialRel=esriSpatialRelEnvelopeIntersects', headers=headers, proxies=proxies).json()

In [5]:
idList = locationIdResponse['objectIds']

In [6]:
from itertools import islice

In [7]:
it = iter(idList) 
n = 100
res = [list(islice(it, n)) for _ in range((len(idList) + n - 1) // n)]

In [8]:
dataset = []
for item in res:
    item = [str(x) for x in item]
    ids = ",".join(item)
    data = session.get(f"https://sigmas.social.gouv.fr/server/rest/services/baignades/fra_vue_baignade/MapServer/5/query?f=geojson&objectIds={ids}&inSR&outSR&returnGeometry=true&outFields=*&returnM=false&returnZ=false", headers=headers, proxies=proxies).json()
    dataset = dataset  + data['features']

In [9]:
import pandas as pd

In [10]:
df = pd.json_normalize(dataset)

In [11]:
df

,type,id,geometry.type,geometry.coordinates,properties.OBJECTID,properties.lsite,properties.lcom,properties.dptddass,properties.isite,properties.annee,properties.class_nat,properties.fvalid,properties.dtdebsai,properties.dtfinsai
0,Feature,1,Point,"[-2.4556977268936504, 47.26728240513849]",1,LA GOVELLE,BATZ-SUR-MER,044,000511,2025,1,O,2025-06-15 00:00:00,2025-09-15 00:00:00
1,Feature,2,Point,"[2.4309229549256335, 48.760449220438844]",2,Plaine sud du Parc de Choisy,CHOISY-LE-ROI,094,000001,2025,1,O,2025-04-01 00:00:00,2025-10-31 00:00:00
2,Feature,3,Point,"[4.5256678719055285, 46.78933051172293]",3,MONTAUBRY LES PATINS,BREUIL (LE),071,001289,2025,1,O,2025-07-01 00:00:00,2025-08-31 00:00:00
3,Feature,4,Point,"[6.852752881493553, 48.06674163518667]",4,RAMBERCHAMP-KATTENDYCKE,GERARDMER,088,004117,2025,1,O,2025-05-29 00:00:00,2025-09-07 00:00:00
4,Feature,5,Point,"[-4.411277068379034, 48.638186410759275]",5,CROIX,GUISSENY,029,001743,2022,11,O,2022-06-15 00:00:00,2022-09-15 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15510,Feature,15511,Point,"[2.2812829935738845, 46.03349971141232]",15511,LA NAUTE,CHAMPAGNAT,023,001206,2026,I,O,2025-07-01 00:00:00,2025-08-31 00:00:00
15511,Feature,15512,Point,"[3.984045972898026, 44.93584254255614]",15512,PLAN D'EAU DU MOULIN DE SAVIN,MONASTIER-SUR-GAZEILLE (LE),043,001594,2026,11,O,2025-07-01 00:00:00,2025-07-31 00:00:00
15512,Feature,15513,Point,"[5.793558481126024, 45.55463862625989]",15513,PLAGE CAMPING BELLEVUE,SAINT-ALBAN-DE-MONTBEL,073,003700,2026,11,O,2025-06-08 00:00:00,2025-08-31 00:00:00
15513,Feature,15514,Point,"[5.796720816827432, 45.56190341311844]",15514,PLAGE SAINT ALBAN,SAINT-ALBAN-DE-MONTBEL,073,003701,2026,11,O,2025-07-01 00:00:00,2025-08-31 00:00:00
